### Exercise 1: Secure User Authentication System

In [2]:
import datetime
import hashlib

#Audit Log
class AuditLog:
    """In-memory audit log for the entire system"""
    _entries = []

    @classmethod
    def write(cls, message):
        timestamp = datetime.datetime.now().isoformat()
        cls._entries.append(f"[{timestamp}] {message}")

    @classmethod
    def show(cls):
        return "\n".join(cls._entries)


#User Class (Secure Authentication)
class User:
    """Represents a system user with authentication, role, and account locking"""

    def __init__(self, username, plain_password, role="standard"):
        if not isinstance(username, str) or not username.strip():
            raise ValueError("Username must be a non-empty string")
        if role not in ("admin", "standard", "guest"):
            raise ValueError("Role must be 'admin', 'standard', or 'guest'")

        self.__username = username.strip()
        self.__role = role
        self.__hashed_password = self.__hash_password(plain_password)
        self.__login_attempts = 0
        self.__account_locked = False

        AuditLog.write(f"User '{self.__username}' created with role '{role}'")

#Private helpers
    def __hash_password(self, plain_password):
        """Hash password using SHA-256"""
        return hashlib.sha256(plain_password.encode()).hexdigest()

#Public getters
    def get_username(self):
        return self.__username

    def get_role(self):
        return self.__role

    def is_admin(self):
        return self.__role == "admin"

    def is_locked(self):
        return self.__account_locked

#Authentication logic
    def authenticate(self, provided_password):
        """Check password, manage attempts, lock after 3 failures"""
        if self.__account_locked:
            AuditLog.write(f"Auth refused - account {self.__username} is locked")
            print(f"Account '{self.__username}' is locked. Contact admin!")
            return False

        provided_hash = self.__hash_password(provided_password)
        if provided_hash == self.__hashed_password:
            self.__login_attempts = 0
            AuditLog.write(f"Successful authentication for {self.__username}")
            return True
        else:
            self.__login_attempts += 1
            AuditLog.write(f"Failed authentication for {self.__username} (attempt #{self.__login_attempts})")
            if self.__login_attempts >= 3:
                self.__account_locked = True
                AuditLog.write(f"Account {self.__username} locked due to 3 failed attempts")
                print(f"Account '{self.__username}' has been locked.")
            return False

    def lock_account(self):
        """Lock the account (can be called by admin or system)"""
        self.__account_locked = True
        AuditLog.write(f"Account {self.__username} locked by system/admin")

    def unlock_account(self, admin_user):
        """Only an admin can unlock an account"""
        if not admin_user.is_admin():
            raise PermissionError("Only admin can unlock accounts")
        self.__account_locked = False
        self.__login_attempts = 0
        AuditLog.write(f"Account {self.__username} unlocked by admin {admin_user.get_username()}")

    def change_privilege(self, new_role, admin_user):
        """Change user role - only admin can do this"""
        if not admin_user.is_admin():
            raise PermissionError("Only admin can change privileges")
        if new_role not in ("admin", "standard", "guest"):
            raise ValueError("Invalid role")
        old_role = self.__role
        self.__role = new_role
        AuditLog.write(f"Privilege for {self.__username} changed from {old_role} to {new_role} by admin {admin_user.get_username()}")

    def get_public_info(self):
        """Return safe, non-sensitive user information"""
        return {
            "username": self.__username,
            "role": self.__role,
            "locked": self.__account_locked
        }


#Demo
if __name__ == "__main__":
#Creating admin and standard user
    admin = User("admin1", "AdminPass123", "admin")
    alice = User("alice", "AliceSecret", "standard")

    print("\nAuthentication tests")
    print(alice.authenticate("wrong"))
    print(alice.authenticate("wrong"))
    print(alice.authenticate("wrong"))
    print(alice.authenticate("AliceSecret"))

    print("\nAdmin unlocks account")
    alice.unlock_account(admin)
    print(alice.authenticate("AliceSecret"))

    print("\nPrivilege escalation (authorised)")
    alice.change_privilege("admin", admin)
    print(f"Alice is now admin? {alice.is_admin()}")

    print("\nPublic info")
    print(admin.get_public_info())
    print(alice.get_public_info())

    print("\nAudit Log")
    print(AuditLog.show())


Authentication tests
False
False
Account 'alice' has been locked.
False
Account 'alice' is locked. Contact admin!
False

Admin unlocks account
True

Privilege escalation (authorised)
Alice is now admin? True

Public info
{'username': 'admin1', 'role': 'admin', 'locked': False}
{'username': 'alice', 'role': 'admin', 'locked': False}

Audit Log
[2026-05-06T10:07:35.718212] User 'admin1' created with role 'admin'
[2026-05-06T10:07:35.718212] User 'alice' created with role 'standard'
[2026-05-06T10:07:35.718212] Failed authentication for alice (attempt #1)
[2026-05-06T10:07:35.718212] Failed authentication for alice (attempt #2)
[2026-05-06T10:07:35.718212] Failed authentication for alice (attempt #3)
[2026-05-06T10:07:35.718212] Account alice locked due to 3 failed attempts
[2026-05-06T10:07:35.718212] Auth refused - account alice is locked
[2026-05-06T10:07:35.718212] Account alice unlocked by admin admin1
[2026-05-06T10:07:35.718212] Successful authentication for alice
[2026-05-06T10:0

### Exercise 2: IoT Device Management System

In [3]:
import datetime
from datetime import timedelta


class AuditLog:
    _entries = []
    @classmethod
    def write(cls, message):
        timestamp = datetime.datetime.now().isoformat()
        cls._entries.append(f"[{timestamp}] {message}")
    @classmethod
    def show(cls):
        return "\n".join(cls._entries)


class User:
    def __init__(self, username, role):
        self.__username = username
        self.__role = role
    def get_username(self):
        return self.__username
    def is_admin(self):
        return self.__role == "admin"
    def get_public_info(self):
        return {"username": self.__username, "role": self.__role}

#IoT Device Class
class IoTDevice:
    """Represents a device with compliance, quarantine, and ownership"""

    def __init__(self, device_id, owner_user, device_type="sensor"):
        if not isinstance(device_id, str) or not device_id.strip():
            raise ValueError("Device ID must be non-empty")
        if not isinstance(owner_user, User):
            raise ValueError("Owner must be a User instance")

        self.__device_id = device_id.strip()
        self.__owner = owner_user
        self.__device_type = device_type
        self.__quarantined = False
        self.__compromised = False
        self.__last_scan_date = datetime.datetime.now()
        self.__firmware_version = "1.0"

        AuditLog.write(f"Device '{self.__device_id}' created, owner {owner_user.get_username()}")

#Getters
    def get_id(self):
        return self.__device_id

    def get_owner(self):
        return self.__owner

    def is_quarantined(self):
        return self.__quarantined

    def is_compromised(self):
        return self.__compromised

#Compliance logic
    def is_compliant(self):
        """Device is compliant if not quarantined, not compromised, and scanned within 30 days"""
        if self.__quarantined or self.__compromised:
            return False
        days_since_scan = (datetime.datetime.now() - self.__last_scan_date).days
        return days_since_scan <= 30

#Authorised actions
    def perform_security_scan(self, performing_user):
        """Only owner or admin can scan. Updates last scan date"""
        if performing_user != self.__owner and not performing_user.is_admin():
            AuditLog.write(f"Security scan denied on {self.__device_id} by {performing_user.get_username()}")
            raise PermissionError("Only owner or admin can scan this device")
        self.__last_scan_date = datetime.datetime.now()
        AuditLog.write(f"Security scan completed on {self.__device_id} by {performing_user.get_username()}")
        print(f"Device {self.__device_id} scan completed. Compliant: {self.is_compliant()}")
        return True

    def update_firmware(self, new_version, performing_user):
        """Only owner or admin can update firmware"""
        if performing_user != self.__owner and not performing_user.is_admin():
            raise PermissionError("Only owner or admin can update firmware")
        self.__firmware_version = new_version
        self.__last_scan_date = datetime.datetime.now()
        AuditLog.write(f"Device {self.__device_id} firmware updated to {new_version} by {performing_user.get_username()}")

    def quarantine(self, performing_user):
        """Only admin can quarantine a device"""
        if not performing_user.is_admin():
            raise PermissionError("Only admin can quarantine devices")
        self.__quarantined = True
        AuditLog.write(f"Device {self.__device_id} quarantined by admin {performing_user.get_username()}")

    def mark_compromised(self, performing_user):
        """Only admin can mark as compromised (auto‑quarantine)"""
        if not performing_user.is_admin():
            raise PermissionError("Only admin can mark device as compromised")
        self.__compromised = True
        self.__quarantined = True
        AuditLog.write(f"Device {self.__device_id} marked compromised by admin {performing_user.get_username()}")

    def override_compliance(self, admin_user):
        """Admin override for compliance checks (logged)"""
        if not admin_user.is_admin():
            return False
        AuditLog.write(f"Admin {admin_user.get_username()} overrode compliance for {self.__device_id}")
        return True

    def can_access(self, requesting_user):
        """
        Determines if a user can access this device
        - Admin always can (but action is logged)
        - Non‑admin must be owner AND device must be compliant
        """
        if requesting_user.is_admin():
            AuditLog.write(f"Admin {requesting_user.get_username()} accessed device {self.__device_id}")
            return True
        if requesting_user != self.__owner:
            AuditLog.write(f"Access denied for {requesting_user.get_username()} (not owner) to {self.__device_id}")
            return False
        if not self.is_compliant():
            AuditLog.write(f"Access denied – device {self.__device_id} non‑compliant for owner {self.__owner.get_username()}")
            return False
        AuditLog.write(f"Access granted to owner {self.__owner.get_username()} for device {self.__device_id}")
        return True

    def get_public_info(self, requesting_user):
        """Only owner or admin can see device details"""
        if requesting_user != self.__owner and not requesting_user.is_admin():
            return {"error": "Access denied"}
        return {
            "device_id": self.__device_id,
            "type": self.__device_type,
            "owner": self.__owner.get_username(),
            "firmware": self.__firmware_version,
            "quarantined": self.__quarantined,
            "compromised": self.__compromised,
            "last_scan": self.__last_scan_date.strftime("%Y-%m-%d"),
            "compliant": self.is_compliant()
        }


#Demo
if __name__ == "__main__":
#Create users
    admin = User("admin_alice", "admin")
    owner = User("bob", "standard")
    stranger = User("eve", "standard")

#Create device owned by 'bob'
    sensor = IoTDevice("TEMP-01", owner, "temperature")

    print("Initial state")
    print(sensor.get_public_info(owner))

    print("\nOwner access (compliant)")
    print(f"Can owner access? {sensor.can_access(owner)}")

#Simulate 31 days without scan
    sensor._IoTDevice__last_scan_date = datetime.datetime.now() - timedelta(days=31)
    print(f"\nAfter 31 days without scan - compliant? {sensor.is_compliant()}")
    print(f"Owner access now? {sensor.can_access(owner)}")

    print("\nAdmin override compliance")
    if sensor.override_compliance(admin):
        print("Admin override applied - but access still requires owner or admin")
        print(f"Admin access: {sensor.can_access(admin)}")

    print("\nPerform security scan (owner)")
    sensor.perform_security_scan(owner)
    print(f"Compliant after scan? {sensor.is_compliant()}")
    print(f"Owner access now? {sensor.can_access(owner)}")

    print("\nQuarantine by admin")
    sensor.quarantine(admin)
    print(sensor.get_public_info(admin))

    print("\nAudit Log (last 5 entries)")
    log_entries = AuditLog.show().split("\n")
    for entry in log_entries[-5:]:
        print(entry)

Initial state
{'device_id': 'TEMP-01', 'type': 'temperature', 'owner': 'bob', 'firmware': '1.0', 'quarantined': False, 'compromised': False, 'last_scan': '2026-05-06', 'compliant': True}

Owner access (compliant)
Can owner access? True

After 31 days without scan - compliant? False
Owner access now? False

Admin override compliance
Admin override applied - but access still requires owner or admin
Admin access: True

Perform security scan (owner)
Device TEMP-01 scan completed. Compliant: True
Compliant after scan? True
Owner access now? True

Quarantine by admin
{'device_id': 'TEMP-01', 'type': 'temperature', 'owner': 'bob', 'firmware': '1.0', 'quarantined': True, 'compromised': False, 'last_scan': '2026-05-06', 'compliant': False}

Audit Log (last 5 entries)
[2026-05-06T10:25:52.980725] Admin admin_alice overrode compliance for TEMP-01
[2026-05-06T10:25:52.980725] Admin admin_alice accessed device TEMP-01
[2026-05-06T10:25:52.982268] Security scan completed on TEMP-01 by bob
[2026-05-0